## Teste de tracking com quadrado simples em fundo branco

In [ ]:
import cv2
import matplotlib.pyplot as plt
import wisardpkg as wp
import matplotlib.patches as patches
import numpy as np

### Capturando o primeiro frame do vídeo

In [ ]:
# Caminho para o vídeo (ajuste conforme sua estrutura)
video_path = "video_simples.mp4"

# Abrir o vídeo
cap = cv2.VideoCapture(video_path)

def get_frame(cap, index):
    cap.set(cv2.CAP_PROP_POS_FRAMES, index)
    ret, frame = cap.read()
    return ret, frame

# Ler o frame 1 e frame 5
ret1, frame1 = get_frame(cap, 1)
ret5, frame5 = get_frame(cap, 2)

# Fechar o vídeo
cap.release()

# Verificar e exibir os frames
if ret1 and ret5:
    # Exibir os dois frames lado a lado
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(frame1)
    plt.title("Frame 1")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(frame5)
    plt.title("Frame 5")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

else:
    print("Erro ao ler um dos frames.")



### Capturando o objeto a partir das coordenadas iniciais

obs: entender e ajustar distorção

In [ ]:
import matplotlib.pyplot as plt
import cv2

# ----- Coordenadas do quadrado -----
x, y = 0, 83
w, h = 35, 35

# ----- Recortar a região de interesse (ROI) -----
roi = frame1[y:y+h, x:x+w]

# ----- Plotar com borda verde -----
fig, ax = plt.subplots(figsize=(3, 3))

# Fundo branco para a área total
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Exibir imagem
ax.imshow(roi)

# Título (opcional)
ax.set_title(f"ROI: x={x}, y={y}, w={w}, h={h}", color='black')

# Mostrar os spines (bordas do gráfico) como verdes
for pos in ["left", "right", "top", "bottom"]:
    ax.spines[pos].set_visible(True)
    ax.spines[pos].set_color("black")
    ax.spines[pos].set_linewidth(2)

# Remover ticks mas manter spines
ax.set_xticks([])
ax.set_yticks([])

plt.show()


### Binarizando

In [ ]:
# Inicializa o binarizador com um limiar de 127
binarizador = wp.Thresholding(127)

# Aplica a binarização
# Achata a imagem para lista de floats
roi_list = roi.flatten().astype(float).tolist()

# Aplica a binarização corretamente
bits = binarizador.transform(roi_list)

# Cria o BinInput
bin_input = wp.BinInput(bits)

In [ ]:
# Criar modelo com base no formato correto
X_train = [bin_input]  # Ex: [[‘1’, ‘0’, ..., ‘1’]]

# y_train: lista com os rótulos correspondentes
y_train = ["quadrado"]

# Criar o DataSet
dataset = wp.DataSet(X_train, y_train)

memory_size = 64       #  numero de RAMs
min_score = 0.01                       # sensibilidade da classificação
threshold = 1                       # pode ajustar com base nos testes
discriminators_limit = 1             # número máximo de classes
model = wp.ClusWisard(memory_size, min_score, threshold, discriminators_limit)
# Treinar
model.train(wp.DataSet(X_train, y_train))

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Suponha que frame2 já foi carregado como frame2 (em BGR)
# Dados do objeto no frame 1
x, y = 0, 83
w, h = 35, 35

# Janela de busca: 3x maior que o objeto, centralizada
search_x = max(0, x - w)
search_y = max(0, y - h)
search_w = 3 * w
search_h = 3 * h

# Fazer uma cópia do frame2 para desenhar
frame5_copy = frame5.copy()

# Desenhar o retângulo da janela de busca
cv2.rectangle(
    frame5_copy,
    (search_x, search_y),
    (search_x + search_w, search_y + search_h),
    color=(0, 255, 0),  # verde
    thickness=2
)

# Converter para RGB para exibir com matplotlib
frame2_rgb = cv2.cvtColor(frame5_copy, cv2.COLOR_BGR2RGB)

# Exibir o frame 2 com a janela desenhada
plt.figure(figsize=(6, 6))
plt.imshow(frame2_rgb)
plt.title("Frame 5 com janela de busca (3x)")
plt.axis("off")
plt.show()


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Tamanho do patch
patch_w, patch_h = w, h  # exemplo: 35x35
stride = 10

# Recorte da janela de busca
search_roi = frame5[search_y:search_y + search_h, search_x:search_x + search_w]

# Armazenar patches e suas posições globais
all_patches = []
all_coords = []

# Percorrer a região de busca com stride definido
for i in range(0, search_roi.shape[0] - patch_h + 1, stride):
    for j in range(0, search_roi.shape[1] - patch_w + 1, stride):
        patch = search_roi[i:i + patch_h, j:j + patch_w]
        all_patches.append(patch)
        all_coords.append((j + search_x, i + search_y))  # coordenadas globais


num_to_show = len(all_patches)
cols = min(num_to_show, 10)  # máx. 10 por linha
rows = (num_to_show + cols - 1) // cols

plt.figure(figsize=(3 * cols, 3 * rows))
for idx in range(num_to_show):
    patch = all_patches[idx]
    gx, gy = all_coords[idx]
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
    plt.title(f"x={gx}, y={gy}", fontsize=8)
    plt.axis("off")

plt.suptitle(f"Todas as {num_to_show} regiões da janela de busca ({patch_w}x{patch_h})")
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import cv2

# Lista para armazenar binarizações
all_bin_inputs = []

for patch in all_patches:
    
    # Inicializa o binarizador com um limiar de 127
    binarizador = wp.Thresholding(127)

    # Aplica a binarização
    # Achata a imagem para lista de floats
    roi_list = roi.flatten().astype(float).tolist()

    # Aplica a binarização corretamente
    bits = binarizador.transform(roi_list)

    # Cria o BinInput
    bin_input = wp.BinInput(bits)
    
    # Adiciona à lista
    all_bin_inputs.append(bin_input)


In [ ]:
# Lista para armazenar os resultados da classificação
classified_labels = []

# Classifica cada patch binarizado
for bin_input in all_bin_inputs:
    prediction = model.classify(bin_input)
    classified_labels.append(prediction)

In [ ]:
print(classified_labels)

In [ ]:
# Índices dos patches classificados como 'quadrado'
quadrado_indices = [i for i, label in enumerate(classified_labels) if label == "quadrado"]

# Coordenadas dos patches classificados como 'quadrado'
quadrado_coords = [all_coords[i] for i in quadrado_indices]

# Patches correspondentes
quadrado_patches = [all_patches[i] for i in quadrado_indices]


In [ ]:
num_to_show = len(quadrado_patches)
cols = 5
rows = (num_to_show + cols - 1) // cols

plt.figure(figsize=(3 * cols, 3 * rows))
for idx, patch in enumerate(quadrado_patches):
    gx, gy = quadrado_coords[idx]
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
    plt.title(f"x={gx}, y={gy}", fontsize=8)
    plt.axis("off")

plt.suptitle(f"{num_to_show} patches classificados como 'quadrado'")
plt.tight_layout()
plt.show()